# 5 비교모델 평가만 60회 — 판정 규칙을 보정 없는 argmax 로(학습은 다시 하지 않음)

**L4 로 실행하십시오.** [셀 1] 이 Blackwell(sm_100·sm_120)이면 멈춥니다. GPU 가 없으면 CPU 로 돕니다(시간 제한 없음).
**'모두 실행' 한 번으로 끝까지 돕니다.** 세션이 끊기면 다시 '모두 실행' — 끝난 회차는 건너뜁니다.
`4 Mark5.3 라벨 없음 0 5회·결과물 4개` 와 동시에 돌려도 됩니다. 노트북 4 의 마지막 셀은 이 60회가 끝나길 기다렸다가 결과물 4개를 만듭니다.

**왜 다시 평가하나 (2026-10-26)**
- 비교모델 eval 은 2026-09-22 부터 val 에서 고른 클래스 편향·others 규칙을 test 에 걸었습니다. 그때는 제안모델도 같은 보정을 썼습니다.
- Mark5.3 은 2026-10-22 에 보정을 빼고 보정 없는 argmax 로 판정합니다. 두 쪽 판정 규칙을 같게 하려고 비교모델 eval 도 보정 없는 argmax 로 바꿨습니다.
- 보정은 답을 고르는 단계에만 걸려 있어 학습 결과는 그대로입니다. 이 노트북은 60회의 **val·test 평가만** 다시 합니다. 학습이 끼어들면 [셀 4] 가 멈춥니다.

| 모델 | 회차 |
|---|---|
| EfficientAT · PANNs · CED-Tiny | 정답 라벨만 × 정답 200·500·1,000·전량 × seed 1~5 = 저장소마다 20회 |

끝나면 저장소마다 `plots/argmax_summary_<모델>_sup_adam_test.csv` 를 다시 만들고, 성능 요약이 argmax 값과 같은지 확인합니다.

In [ ]:
#========================================================================
# [셀 1] 런타임 확인 — L4 권장 · Blackwell(sm_100·120)이면 멈춤 · CPU 는 허용(멈추지 않음)
#========================================================================
import subprocess, sys
_probe = subprocess.run([sys.executable, "-c",
    "import os, torch\n"
    "ok = torch.cuda.is_available()\n"
    "cap = torch.cuda.get_device_capability(0) if ok else None\n"
    "print('GPU', torch.cuda.get_device_name(0) if ok else '없음 — CPU 로 돈다')\n"
    "print('SM', cap[0] * 10 + cap[1] if cap else 'cpu')\n"
    "print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| CPU 코어', os.cpu_count())"],
    capture_output=True, text=True)
print(_probe.stdout, _probe.stderr[-2000:])
_sm = next((l.split()[1] for l in _probe.stdout.splitlines() if l.startswith("SM ")), "cpu")
if _sm in ("100", "120"):
    raise SystemExit("[STOP] Blackwell(sm_100/sm_120) 런타임입니다. L4 로 바꿔 다시 '모두 실행' 하십시오(2026-10-17 G4 계산 오류).")
!nvidia-smi --query-gpu=name,driver_version,compute_cap,memory.total --format=csv 2>/dev/null || echo "nvidia-smi 없음(CPU 런타임)"
!free -g | head -2

In [ ]:
#========================================================================
# [셀 2] 구글 드라이브 마운트
#========================================================================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#========================================================================
# [셀 3] 코드 판 확인 + 데이터 목록 — Mark5.2 의 목록과 같게 맞추고 행 수 확인
#========================================================================
import csv, json, os, re, time
EXP = "/content/drive/MyDrive/NewProject/codes/experiments"
DATA_ROOT = f"{EXP}/Mark5.2"
REPOS = {"Baseline_EfficientAT_mn10": "efficientat_mn10", "Baseline_PANNs_CNN6": "panns_cnn6", "Baseline_CED_Tiny": "ced_tiny"}
NEED = {"common/naming.py": ['OPTIMIZER_NAME = "adam"', "def adam_lr"],   # [2026-10-25 / A안] Adam · 선정값 고정 · run_tag _adam
        "common/train_loop.py": ["optimizer = Adam("],
        # [2026-10-26] 판정 규칙 = 보정 없는 argmax(Mark5.3 과 같음) · 옛 val 선택 json 이면 val·test 를 다시 평가
        "common/others_rule.py": ['DECISION_POLICY = "argmax_v1"', "def save_argmax_json", "def rule_json_is_current"],
        "eval.py": ["save_argmax_json(rule_json", "옛 val 선택 규칙", "옛 평가 결과 삭제(다시 평가)"],
        "common/metrics.py": ["예측 CSV 를 성능 요약보다 먼저 쓴다"],
        "run_experiments.py": ['rule_json_is_current(a["rule"], tag, a["ckpt"])', '"--eval_only"'],
        "make_argmax_summary.py": ["def main", "PreBias_Prob_"]}
for repo in REPOS:
    root = f"{EXP}/{repo}"
    miss = [(f, s) for f, ss in NEED.items() for s in ss
            if not os.path.isfile(f"{root}/{f}") or s not in open(f"{root}/{f}", encoding="utf-8").read()]
    if miss:
        raise SystemExit(f"[STOP] {repo} 코드가 10-26(판정 규칙 argmax) 판이 아닙니다: {miss}")
    for name in ("dataset_labeled.csv", "dataset_unlabeled.csv", "dataset_val.csv", "dataset_test.csv"):
        src, dst = f"{DATA_ROOT}/{name}", f"{root}/{name}"
        if open(src, "rb").read() != (open(dst, "rb").read() if os.path.isfile(dst) else b""):
            open(dst, "wb").write(open(src, "rb").read())
            print(f"[복사] {repo}/{name} <- Mark5.2")
    rows = {n: sum(1 for _ in csv.DictReader(open(f"{root}/{n}", encoding="utf-8-sig")))
            for n in ("dataset_labeled.csv", "dataset_unlabeled.csv", "dataset_val.csv", "dataset_test.csv")}
    if rows != {"dataset_labeled.csv": 2000, "dataset_unlabeled.csv": 2000, "dataset_val.csv": 430, "dataset_test.csv": 430}:
        raise SystemExit(f"[STOP] {repo} 데이터 목록 행 수 {rows}")
    print(f"{repo}: 코드 판 확인 · 데이터 목록 {rows}")

In [ ]:
#========================================================================
# [셀 3b] 패키지 — PANNs 전용 torchlibrosa(코랩 기본 환경에 없음). 2026-10-25 에는 이 셀이 없어 PANNs 회차가 시작 전에 실패했다
#========================================================================
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchlibrosa==0.1.0"], check=True)
_chk = subprocess.run([sys.executable, "-c", "import model_def, importlib.metadata as m; print('torchlibrosa', m.version('torchlibrosa'))"],
                      capture_output=True, text=True, cwd=f"{EXP}/Baseline_PANNs_CNN6")
print(_chk.stdout, _chk.stderr[-2000:])
if _chk.returncode != 0 or "torchlibrosa 0.1.0" not in _chk.stdout:
    raise SystemExit("[STOP] PANNs 모델 코드를 불러오지 못했습니다(torchlibrosa 0.1.0) — 위 오류 확인 뒤 다시 '모두 실행'")

In [ ]:
#========================================================================
# [셀 4] 계획 확인 — 저장소마다 20줄 · 평가만(학습이 끼면 멈춤) · 체크포인트가 비어 있지 않은가
#========================================================================
import subprocess
B_RUN = "--phase main --conds sup --labeled_ns 0 1000 500 200 --seeds 1 2 3 4 5 --eval_only"   # 평가만


def plan(repo):
    p = subprocess.run(f"{sys.executable} run_experiments.py {B_RUN} --data_root {DATA_ROOT} --dry_run",
                       shell=True, capture_output=True, text=True, cwd=f"{EXP}/{repo}")
    rows = [l.strip() for l in p.stdout.splitlines() if "[실행]" in l or "[건너뜀]" in l]
    if p.returncode != 0 or len(rows) != 20 or not all("_adam_" in r for r in rows):
        print(p.stdout[-3000:]); print(p.stderr[-3000:])
        raise SystemExit(f"[STOP] {repo} 계획을 읽지 못했습니다(종료 코드 {p.returncode} · {len(rows)}줄, 기대 20줄)")
    return rows


for repo, mtag in REPOS.items():
    rows = plan(repo)
    train_rows = [r for r in rows if "[실행]" in r and "학습" in r]
    if train_rows:
        print("\n".join(train_rows))
        _p = subprocess.run(f"{sys.executable} run_experiments.py {B_RUN} --data_root {DATA_ROOT} --dry_run",
                            shell=True, capture_output=True, text=True, cwd=f"{EXP}/{repo}")
        print("\n".join(l for l in _p.stdout.splitlines() if "[WARN]" in l or "[FAIL]" in l)[-3000:])   # 학습이 필요하다고 본 이유
        raise SystemExit(f"[STOP] {repo}: 학습부터 다시 하려는 회차가 {len(train_rows)}개입니다 — 이 노트북은 평가만 다시 합니다. "
                         "체크포인트·loss_history·train_meta 가 드라이브에 있는지 확인하십시오.")
    empty = [t for t in (f"{mtag}_sup_adam" + ("" if l is None else f"_l{l}") + f"_s{s}" for l in (200, 500, 1000, None) for s in range(1, 6))
             if not os.path.isfile(f"{EXP}/{repo}/model/{t}.pth") or os.path.getsize(f"{EXP}/{repo}/model/{t}.pth") == 0]
    if empty:
        raise SystemExit(f"[STOP] {repo}: 체크포인트가 없거나 비어 있는 회차 {empty} — 다시 평가할 수 없습니다")
    print(f"{repo}: 다시 평가할 회차 {sum('[실행]' in r for r in rows)} / 20 (학습 0회)")

In [ ]:
#========================================================================
# [셀 5] 비교모델 ★ 3 × 20회 평가만 — val(판정 규칙 json 기록) → test. 끝난 회차는 건너뛴다. 시간 제한 없음
#========================================================================
for repo in REPOS:
    print(f"\n######## {repo}  정답 라벨만 × 정답 전량·1,000·500·200 × seed 1~5 — 평가만(argmax)", flush=True)
    !cd "{EXP}/{repo}" && python run_experiments.py --phase main --conds sup --labeled_ns 0 1000 500 200 --seeds 1 2 3 4 5 --eval_only --data_root "{DATA_ROOT}"

In [ ]:
#========================================================================
# [셀 6] 확인 — 60회 평가가 끝났는가 · 판정 규칙 json 이 전부 argmax_v1 인가 · 성능 요약 = argmax 값인가 (argmax 표 다시 만들기)
#========================================================================
import statistics
left = {repo: [r for r in plan(repo) if "[실행]" in r] for repo in REPOS}
if any(left.values()):
    for repo, rows in left.items():
        for r in rows:
            print(repo, r)
    raise SystemExit(f"[STOP] 남은 회차 {sum(map(len, left.values()))}개 — 셀 5 출력의 [FAIL] 을 확인하고 다시 '모두 실행'")
METRICS = ("Accuracy", "Macro Precision", "Macro Recall", "Macro F1")
for repo, mtag in REPOS.items():
    tags = [f"{mtag}_sup_adam" + ("" if l is None else f"_l{l}") + f"_s{s}" for l in (200, 500, 1000, None) for s in range(1, 6)]
    bad = [t for t in tags if json.load(open(f"{EXP}/{repo}/plots/others_threshold_{t}.json", encoding="utf-8")).get("decision_policy") != "argmax_v1"]
    if bad:
        raise SystemExit(f"[STOP] {repo}: 판정 규칙 json 이 argmax_v1 이 아닌 회차 {bad}")
    p = subprocess.run([sys.executable, "make_argmax_summary.py"], capture_output=True, text=True, cwd=f"{EXP}/{repo}")
    print(p.stdout.strip()[-500:], p.stderr.strip()[-1500:])
    if p.returncode != 0:
        raise SystemExit(f"[STOP] {repo}: make_argmax_summary.py 실패")
    rows = list(csv.DictReader(open(f"{EXP}/{repo}/plots/argmax_summary_{mtag}_sup_adam_test.csv", encoding="utf-8")))
    worst = max(abs(float(r[f"argmax-reported {m}"])) for r in rows for m in METRICS)
    if len(rows) != 20 or worst > 1e-6:
        raise SystemExit(f"[STOP] {repo}: 성능 요약이 argmax 값과 다릅니다(최대 차이 {worst:g}, {len(rows)}행)")
    print(f"\n{repo}: 20회 모두 판정 규칙 argmax_v1 · 성능 요약 = argmax 값(최대 차이 {worst:g})")
    for l in ("200", "500", "1000", "2000"):
        v = [r for r in rows if r["labeled"] == l]
        print(f"  정답 {'전량' if l == '2000' else l} (n={len(v)}): " + " · ".join(f"{m} {statistics.mean(float(r['argmax ' + m]) for r in v):.4f}" for m in METRICS))
print("\n60회 평가 완료 — 결과물 4개는 '4 Mark5.3 라벨 없음 0 5회·결과물 4개' 노트북의 마지막 셀이 만듭니다.")